# Biorepository Data Cleaning and Merging

This notebook demonstrates common data wrangling tasks for biorepository data, including:

1. **Loading** synthetic datasets representing patients, biospecimen samples, and storage locations
2. **Cleaning** each dataset (handling missing values, standardising formats, removing duplicates)
3. **Merging** the datasets into a single analysis-ready table
4. **Summarising** the merged data


In [ ]:
import numpy as np
import pandas as pd

# Display all columns when printing DataFrames
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

## 1  Create Synthetic Biorepository Datasets

In a real scenario these tables would be loaded from a LIMS export, CSV files, or a database.  
Here we build them inline so the notebook is fully self-contained.

### Dataset A – Patient / Subject Demographics

In [ ]:
patients_raw = pd.DataFrame({
    'patient_id':  ['P001', 'P002', 'P003', 'P004', 'P005', 'P002'],   # P002 is duplicated
    'first_name':  ['Alice', 'Bob', 'carol', 'DAVE', 'Eve', 'Bob'],     # inconsistent casing
    'last_name':   ['Smith', 'Jones', 'Williams', 'Brown', None, 'Jones'],
    'dob':         ['1980-03-15', '1975-07-22', '1990-11-01', '1965-05-30', '2000-09-14', '1975-07-22'],
    'sex':         ['F', 'M', 'f', 'male', 'Female', 'M'],              # inconsistent values
    'diagnosis':   ['Type 2 Diabetes', 'Hypertension', 'Healthy', 'Type 2 Diabetes', None, 'Hypertension'],
})

print('Raw patients table:')
print(patients_raw.to_string(index=False))

### Dataset B – Biospecimen Sample Inventory

In [ ]:
samples_raw = pd.DataFrame({
    'sample_id':       ['S001', 'S002', 'S003', 'S004', 'S005', 'S006', 'S007'],
    'patient_id':      ['P001', 'P001', 'P002', 'P003', 'P004', 'P005', 'P999'],  # P999 has no patient record
    'collection_date': ['2024-01-10', '2024-01-10', '2024-02-14', '2024-03-05',
                        '2024-03-20', '2024-04-01', '2024-04-15'],
    'sample_type':     ['Serum', 'PBMC', 'serum', 'Whole Blood', 'Serum', 'pbmc', 'Serum'],
    'volume_ml':       [2.0, 1.5, None, 5.0, 2.5, 1.0, 2.0],           # one missing volume
    'notes':           [None, 'Haemolysed', None, None, 'Low quality', None, None],
})

print('Raw samples table:')
print(samples_raw.to_string(index=False))

### Dataset C – Freezer Storage Locations

In [ ]:
storage_raw = pd.DataFrame({
    'sample_id':    ['S001', 'S002', 'S003', 'S004', 'S005', 'S006', 'S003'],  # S003 duplicated
    'freezer_id':   ['FRZ-A', 'FRZ-A', 'FRZ-B', 'FRZ-B', 'FRZ-A', 'FRZ-C', 'FRZ-B'],
    'rack':         [1, 1, 2, 2, 1, 3, 2],
    'box':          [1, 2, 1, 2, 3, 1, 1],
    'position':     ['A1', 'B3', 'C5', 'D2', 'A4', 'B1', 'C5'],
    'storage_date': ['2024-01-11', '2024-01-11', '2024-02-15', '2024-03-06',
                     '2024-03-21', '2024-04-02', '2024-02-15'],
})

print('Raw storage table:')
print(storage_raw.to_string(index=False))

## 2  Clean Each Dataset

### 2.1  Clean Patient Data

In [ ]:
patients = patients_raw.copy()

# --- Remove duplicate rows (keep first occurrence) ---
n_before = len(patients)
patients = patients.drop_duplicates(subset='patient_id', keep='first')
print(f'Duplicate patient rows removed: {n_before - len(patients)}')

# --- Standardise name casing ---
patients['first_name'] = patients['first_name'].str.title()
patients['last_name']  = patients['last_name'].str.title()

# --- Standardise sex field to M / F ---
sex_map = {'f': 'F', 'female': 'F', 'male': 'M', 'm': 'M'}
patients['sex'] = patients['sex'].str.lower().map(sex_map).fillna(patients['sex'])

# --- Convert date of birth to datetime ---
patients['dob'] = pd.to_datetime(patients['dob'])

# --- Fill missing diagnosis with 'Unknown' ---
patients['diagnosis'] = patients['diagnosis'].fillna('Unknown')

# --- Fill missing last name with 'Unknown' ---
patients['last_name'] = patients['last_name'].fillna('Unknown')

# --- Compute age in years ---
today = pd.Timestamp('2024-12-31')
patients['age'] = ((today - patients['dob']).dt.days / 365.25).astype(int)

print('\nCleaned patients table:')
print(patients.to_string(index=False))

### 2.2  Clean Sample Data

In [ ]:
samples = samples_raw.copy()

# --- Standardise sample_type to title-case ---
samples['sample_type'] = samples['sample_type'].str.title()

# --- Convert collection_date to datetime ---
samples['collection_date'] = pd.to_datetime(samples['collection_date'])

# --- Impute missing volume with the median volume for that sample type ---
median_vol = samples.groupby('sample_type')['volume_ml'].transform('median')
n_missing = samples['volume_ml'].isna().sum()
samples['volume_ml'] = samples['volume_ml'].fillna(median_vol)
print(f'Missing volume values imputed: {n_missing}')

# --- Flag low-quality samples ---
samples['quality_flag'] = samples['notes'].notna()

# --- Drop the free-text notes column (captured in quality_flag) ---
samples = samples.drop(columns='notes')

print('\nCleaned samples table:')
print(samples.to_string(index=False))

### 2.3  Clean Storage Data

In [ ]:
storage = storage_raw.copy()

# --- Remove duplicate storage records (same sample stored twice in error) ---
n_before = len(storage)
storage = storage.drop_duplicates(subset='sample_id', keep='first')
print(f'Duplicate storage rows removed: {n_before - len(storage)}')

# --- Convert storage_date to datetime ---
storage['storage_date'] = pd.to_datetime(storage['storage_date'])

# --- Create a human-readable location string ---
storage['location'] = (storage['freezer_id'] + ' / Rack ' +
                       storage['rack'].astype(str) + ' / Box ' +
                       storage['box'].astype(str) + ' / ' +
                       storage['position'])

print('\nCleaned storage table:')
print(storage.to_string(index=False))

## 3  Merge Datasets

We perform **two left joins**:

1. `samples` ← `patients` on `patient_id`  (keeps all samples; attaches patient demographics where available)
2. Result ← `storage` on `sample_id`       (attaches storage location where available)

In [ ]:
# Step 1: Join sample inventory with patient demographics
merged = samples.merge(
    patients[['patient_id', 'first_name', 'last_name', 'sex', 'age', 'diagnosis']],
    on='patient_id',
    how='left'
)

# Step 2: Join with storage information
merged = merged.merge(
    storage[['sample_id', 'freezer_id', 'location', 'storage_date']],
    on='sample_id',
    how='left'
)

# Reorder columns for readability
merged = merged[[
    'sample_id', 'patient_id', 'first_name', 'last_name',
    'sex', 'age', 'diagnosis',
    'sample_type', 'collection_date', 'volume_ml', 'quality_flag',
    'freezer_id', 'location', 'storage_date'
]]

print('Merged biorepository table:')
print(merged.to_string(index=False))

## 4  Validate the Merged Data

Check for samples that could not be linked to a patient record or a storage location.

In [ ]:
unmatched_patients = merged[merged['first_name'].isna()]
unmatched_storage  = merged[merged['freezer_id'].isna()]

print(f'Samples with no matching patient record : {len(unmatched_patients)}')
if len(unmatched_patients):
    print(unmatched_patients[['sample_id', 'patient_id']].to_string(index=False))

print(f'\nSamples with no storage record : {len(unmatched_storage)}')
if len(unmatched_storage):
    print(unmatched_storage[['sample_id']].to_string(index=False))

## 5  Summary Statistics

### 5.1  Sample counts by type and diagnosis

In [ ]:
# Only count samples that could be linked to a patient
linked = merged[merged['first_name'].notna()].copy()

pivot = (
    linked
    .groupby(['diagnosis', 'sample_type'])
    .agg(count=('sample_id', 'count'),
         total_volume_ml=('volume_ml', 'sum'),
         flagged=('quality_flag', 'sum'))
    .reset_index()
)

print('Sample counts by diagnosis and type:')
print(pivot.to_string(index=False))

### 5.2  Freezer utilisation

In [ ]:
freezer_util = (
    merged[merged['freezer_id'].notna()]
    .groupby('freezer_id')
    .agg(samples_stored=('sample_id', 'count'),
         total_volume_ml=('volume_ml', 'sum'))
    .reset_index()
)

print('Freezer utilisation:')
print(freezer_util.to_string(index=False))

### 5.3  Overall data quality report

In [ ]:
total_samples       = len(merged)
linked_to_patient   = merged['first_name'].notna().sum()
linked_to_storage   = merged['freezer_id'].notna().sum()
quality_flagged     = merged['quality_flag'].sum()

report = pd.DataFrame({
    'Metric': [
        'Total samples',
        'Linked to patient record',
        'Linked to storage location',
        'Quality-flagged samples',
    ],
    'Count': [total_samples, linked_to_patient, linked_to_storage, quality_flagged],
    '%': [
        '100%',
        f'{100*linked_to_patient/total_samples:.0f}%',
        f'{100*linked_to_storage/total_samples:.0f}%',
        f'{100*quality_flagged/total_samples:.0f}%',
    ]
})

print('Data quality report:')
print(report.to_string(index=False))